## 🎯 Learning Objectives
* Understand the challenges of serving finetuned Large Language Models (LLMs) efficiently in production.
* Learn how vLLM optimizes LLM inference for high throughput and low latency.
* Explore Ollama as a user-friendly solution for local and edge deployment of LLMs.
* Implement client-side code to interact with LLMs served by vLLM and Ollama.
* Evaluate the trade-offs between vLLM and Ollama for different deployment scenarios.


## Serving Finetuned LLMs: vLLM vs. Ollama

Deploying Large Language Models (LLMs), especially finetuned ones, into production environments presents unique challenges. Unlike traditional web services, LLMs are massive, memory-hungry, and computationally intensive. Efficient serving requires optimizing for several key metrics:

1.  **Throughput:** The number of requests processed per unit of time.
2.  **Latency:** The time taken to generate a response for a single request.
3.  **Memory Usage:** Minimizing GPU/CPU memory footprint to serve more models or larger models.

Traditional serving methods often struggle with these demands, leading to high operational costs and slow response times. This is where specialized LLM serving frameworks like **vLLM** and **Ollama** come into play.

### vLLM: The High-Performance Powerhouse

vLLM is an open-source library designed for high-throughput and low-latency LLM inference. It's built by researchers from UC Berkeley and is widely adopted for production-grade deployments. Its core innovation lies in its **PagedAttention** algorithm, which efficiently manages KV (Key-Value) cache memory, significantly reducing memory waste and increasing throughput. Think of it like a highly optimized restaurant kitchen: instead of preparing each dish from scratch for every customer, PagedAttention intelligently reuses ingredients (KV cache) and manages multiple orders (requests) concurrently, ensuring no chef (GPU core) is idle and every customer gets their meal faster.

**Key Features of vLLM:**
*   **PagedAttention:** Efficiently manages the KV cache, leading to higher throughput.
*   **Continuous Batching:** Processes requests as soon as they arrive, rather than waiting for a full batch, reducing latency.
*   **Optimized CUDA Kernels:** Leverages custom kernels for maximum GPU utilization.
*   **Support for LoRA/PEFT Adapters:** Seamlessly loads and serves finetuned models with minimal overhead.
*   **OpenAI-compatible API:** Easy integration with existing LLM applications.

### Ollama: The Local & Accessible Champion

Ollama, on the other hand, focuses on making LLMs accessible for local and edge deployments. It provides a simple command-line interface and an API to run open-source models, including finetuned ones, on your personal machine or smaller servers. It's like having a personal AI assistant that's incredibly easy to set up and use, even if you're not a deep learning expert. Ollama handles the complexities of model quantization, downloading, and running, abstracting away the underlying infrastructure.

**Key Features of Ollama:**
*   **Ease of Use:** Simple CLI for pulling, running, and managing models.
*   **Local Deployment:** Optimized for running models on consumer-grade hardware (CPU and GPU).
*   **Modelfiles:** Allows users to create custom models, including finetuned ones, by defining a `Modelfile`.
*   **OpenAI-compatible API:** Provides a familiar interface for developers.
*   **GGUF Support:** Leverages the efficient GGUF format for quantized models, reducing memory footprint.

### Serving Finetuned Models

Both vLLM and Ollama are excellent choices for serving finetuned models. For instance, if you've finetuned a Llama-3 model using LoRA, vLLM can load the base model and apply your LoRA adapter on the fly, benefiting from its high-performance optimizations. With Ollama, you would typically convert your finetuned model into the GGUF format and then create a `Modelfile` to serve it locally.

In the following sections, we'll demonstrate how to interact with both vLLM and Ollama, assuming their respective servers are running, to serve a finetuned (or base) LLM.


In [ ]:
# Ensure you have 'requests' installed for API interaction
# pip install requests

import requests
import json
import time

# --- Configuration --- #
# Replace with your actual finetuned model path or Hugging Face ID
# For demonstration, we'll use a small, publicly available model.
# If you have a finetuned LoRA adapter, vLLM can load it via `peft_model_id`.
# For Ollama, you'd typically convert your finetuned model to GGUF and create a Modelfile.

# Example model for vLLM (replace with your finetuned model if available)
vllm_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# If you have a LoRA adapter, specify its Hugging Face ID or local path
vllm_peft_adapter_id = None # e.g., "your_hf_user/your_finetuned_lora_adapter"

# Example model for Ollama (replace with your custom model name if available)
ollama_model_name = "llama3:8b-instruct-q4_0" # A common Ollama model
# If you have a custom finetuned model in Ollama, use its name here.

# --- vLLM Server Interaction --- #
print("--- Interacting with vLLM Server ---")

# To run the vLLM server, execute the following command in your terminal:
# (Ensure vLLM is installed: pip install vllm)
# For a base model:
# python -m vllm.entrypoints.api_server --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --port 8000
# For a finetuned model with LoRA adapter:
# python -m vllm.entrypoints.api_server --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --peft_model_id your_hf_user/your_finetuned_lora_adapter --port 8000

vllm_api_url = "http://localhost:8000/v1/completions"

def query_vllm(prompt, max_tokens=128, temperature=0.7):
    headers = {"Content-Type": "application/json"}
    data = {
        "model": vllm_model_id, # This is often ignored by vLLM server if --model is specified at startup
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": False
    }
    try:
        start_time = time.time()
        response = requests.post(vllm_api_url, headers=headers, json=data)
        response.raise_for_status() # Raise an exception for HTTP errors
        end_time = time.time()
        print(f"vLLM Response Time: {end_time - start_time:.2f} seconds")
        return response.json()
    except requests.exceptions.ConnectionError:
        print("Error: vLLM server not running or unreachable. Please start it using the command above.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error querying vLLM: {e}")
        return None

vllm_prompt = "Explain the concept of PagedAttention in vLLM in a concise way."
vllm_response = query_vllm(vllm_prompt)

if vllm_response:
    print("\n--- vLLM Output ---")
    print(json.dumps(vllm_response, indent=2))
    if vllm_response.get("choices"):
        print("\nGenerated Text (vLLM):")
        print(vllm_response["choices"][0]["text"].strip())


# --- Ollama Server Interaction --- #
print("\n--- Interacting with Ollama Server ---")

# To run Ollama, first download and install it from ollama.com.
# Then, in your terminal, pull a model (e.g., ollama pull llama3:8b-instruct-q4_0)
# Ollama automatically starts its API server on port 11434 when you run `ollama serve`
# or when you interact with it via `ollama run`.

# To create a custom finetuned model in Ollama:
# 1. Convert your finetuned model to GGUF format (e.g., using llama.cpp tools).
# 2. Create a Modelfile (e.g., `Modelfile_finetuned`):
#    FROM ./path/to/your_finetuned_model.gguf
#    PARAMETER temperature 0.7
#    TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
#    You are a helpful AI assistant.<|eot_id|>
#    <|start_header_id|>user<|end_header_id|>
#    {{ .Prompt }}<|eot_id|>
#    <|start_header_id|>assistant<|end_header_id|>
#    """
# 3. Create the model: `ollama create your_finetuned_model_name -f Modelfile_finetuned`
# 4. Then use `your_finetuned_model_name` in `ollama_model_name` below.

ollama_api_url = "http://localhost:11434/api/generate"

def query_ollama(prompt, model_name, max_tokens=128, temperature=0.7):
    headers = {"Content-Type": "application/json"}
    data = {
        "model": model_name,
        "prompt": prompt,
        "options": {
            "num_predict": max_tokens,
            "temperature": temperature
        },
        "stream": False
    }
    try:
        start_time = time.time()
        response = requests.post(ollama_api_url, headers=headers, json=data)
        response.raise_for_status()
        end_time = time.time()
        print(f"Ollama Response Time: {end_time - start_time:.2f} seconds")
        return response.json()
    except requests.exceptions.ConnectionError:
        print("Error: Ollama server not running or unreachable. Please ensure Ollama is installed and running.")
        print("Try `ollama serve` in a terminal or `ollama run llama3` to start it.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error querying Ollama: {e}")
        return None

ollama_prompt = "What are the main advantages of using Ollama for local LLM deployment?"
ollama_response = query_ollama(ollama_prompt, ollama_model_name)

if ollama_response:
    print("\n--- Ollama Output ---")
    print(json.dumps(ollama_response, indent=2))
    if ollama_response.get("response"):
        print("\nGenerated Text (Ollama):")
        print(ollama_response["response"].strip())


### Interpreting the Code and Outputs

The code above demonstrates how to interact with both vLLM and Ollama servers using standard HTTP requests. For both, we define a `query_vllm` and `query_ollama` function that sends a JSON payload to their respective API endpoints and prints the response.

**vLLM Output Interpretation:**

The vLLM API typically returns a JSON object with a structure similar to OpenAI's API. Key fields include:
*   `id`: A unique identifier for the completion request.
*   `choices`: A list of generated completions. Each choice contains:
    *   `text`: The generated text from the model.
    *   `index`: The index of the choice.
    *   `logprobs`: (Optional) Log probabilities of the generated tokens.
    *   `finish_reason`: Why the generation stopped (e.g., `length`, `stop`).
*   `usage`: Information about token counts (prompt tokens, completion tokens, total tokens).

When serving a finetuned model with vLLM, you would typically start the vLLM server with the base model and specify your LoRA adapter using the `--peft_model_id` argument. vLLM transparently applies the adapter during inference, allowing you to query the combined model as if it were a single entity.

**Ollama Output Interpretation:**

Ollama's API also returns a JSON object, often simpler than vLLM's for basic generation:
*   `model`: The name of the model used.
*   `created_at`: Timestamp of the generation.
*   `response`: The generated text from the model.
*   `done`: A boolean indicating if the generation is complete.
*   `total_duration`, `load_duration`, `prompt_eval_duration`, `eval_count`, `eval_duration`: Detailed timing and token count information, useful for performance analysis.

For finetuned models in Ollama, you would first convert your model to the GGUF format (a highly optimized format for CPU/GPU inference, often used with `llama.cpp`). Then, you create a `Modelfile` that points to your GGUF file and defines any custom parameters or chat templates. Once created, you can refer to your custom model by its name in API calls.

### Performance Trade-offs and Use Cases

| Feature           | vLLM                                      | Ollama                                    |
| :---------------- | :---------------------------------------- | :---------------------------------------- |
| **Primary Goal**  | Maximize throughput & minimize latency    | Ease of use, local/edge deployment        |
| **Hardware Focus**| High-end GPUs (NVIDIA)                    | CPU & Consumer GPUs (NVIDIA, AMD, Apple Silicon) |
| **Key Tech**      | PagedAttention, Continuous Batching, CUDA | GGUF, Modelfiles, Simplified CLI          |
| **Finetuning**    | Direct LoRA/PEFT adapter loading          | GGUF conversion + Modelfile               |
| **Setup**         | More involved, often Docker/Kubernetes    | Simple CLI installation, single binary    |
| **Scalability**   | Excellent for large-scale production      | Good for local, prototyping, small-scale  |
| **API**           | OpenAI-compatible                         | OpenAI-compatible                         |

**When to use vLLM:**
*   **High-traffic production APIs:** When serving millions of requests per day and requiring maximum GPU utilization.
*   **Real-time applications:** Scenarios where low latency is critical (e.g., chatbots, interactive AI agents).
*   **Cost optimization:** When you need to squeeze the most performance out of expensive GPU hardware.
*   **Complex model architectures:** Supports a wide range of Hugging Face models and advanced features.

**When to use Ollama:**
*   **Local development and prototyping:** Quickly test and iterate on models on your workstation.
*   **Edge deployments:** Running LLMs on devices with limited resources or offline capabilities.
*   **Personal AI assistants:** Setting up a local LLM for personal use without cloud dependencies.
*   **Simplified deployment:** When ease of setup and management is prioritized over raw, peak performance.
*   **CPU-only inference:** Excellent for running models on CPUs when GPUs are unavailable or too costly.

Both tools represent significant advancements in making LLM serving more efficient and accessible. The choice between them depends heavily on your specific deployment needs, hardware availability, and performance requirements.


### Resources

*   **vLLM Documentation:** [https://docs.vllm.ai/en/latest/](https://docs.vllm.ai/en/latest/)
*   **Ollama Website & Documentation:** [https://ollama.com/](https://ollama.com/)
*   **Hugging Face PEFT Library (for LoRA):** [https://huggingface.co/docs/peft/en/index](https://huggingface.co/docs/peft/en/index)
*   **Hugging Face Transformers Library:** [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **llama.cpp (for GGUF conversion):** [https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)
